# Sequence: request/response flows between participants

`d3.sequence()` draws sequence diagrams from a Mermaid `sequenceDiagram` or a DataFrame of
messages. With `notebook=True` the interactive chart is shown inline (isolated in an iframe):
hover a participant to highlight its lifeline and messages, search participants and message
text, use **▶ Walk through** to step through the messages with a caption per step, or open the
side panels for layout, legend and export.

In [ ]:
import pandas as pd
from d3blocks import D3Blocks

# frame=False keeps node_properties as a dict, so entries can be edited by name below.
d3 = D3Blocks(frame=False, verbose='error')

## 1. Mermaid sequence diagram

`participant A as Label`, `actor A`, calls (`->>`), replies (`-->>`) and `rect rgb(...)` regions
are converted. The input can also be Markdown text with a ` ```mermaid ` block or the path to a
`.md` / `.mmd` file. Unsupported statements raise a `ValueError` naming the line.

In [ ]:
oauth = """
sequenceDiagram
    actor Operator
    participant UI as Trade App (UI)
    participant API as Trade API
    participant DB as Database
    participant Schwab as Schwab OAuth

    rect rgb(236, 254, 255)
        Operator->>UI: Click "Connect Schwab"
        UI->>API: GET /brokers/schwab/authorize
        API->>DB: Store state nonce + PKCE verifier
        API-->>UI: Redirect to Schwab authorization URL
    end
    rect rgb(254, 243, 199)
        UI->>Schwab: Redirect browser to authorization URL
        Operator->>Schwab: Approve connection
        Schwab->>API: GET /callback?code=…&state=…
    end
    rect rgb(220, 252, 231)
        API->>Schwab: POST token exchange (code + PKCE verifier)
        Schwab-->>API: access_token + refresh_token
        API->>DB: Store encrypted tokens
        API-->>UI: Settings shows "Connected"
    end
"""
d3.sequence(oauth, notebook=True, filepath=None, figsize=[None, 650])

## 2. Messages from a DataFrame, with a captioned walkthrough

Rows are messages in time order. `style` is `sync` (call) or `reply`; `variant` (`emphasis`,
`security`, `async`, `error`) sets the colour as in flowgraph. `groups` adds regions by message
index, and `steps` replaces the default one-step-per-message walkthrough with your own captions.

In [ ]:
df = pd.DataFrame([
    ('user', 'web', 'open dashboard', 'sync', 'default'),
    ('web', 'api', 'GET /dashboard', 'sync', 'emphasis'),
    ('api', 'auth', 'verify JWT', 'sync', 'security'),
    ('auth', 'api', 'claims ok', 'reply', 'default'),
    ('api', 'cache', 'read cache', 'sync', 'default'),
    ('cache', 'api', 'miss', 'reply', 'default'),
    ('api', 'db', 'query profile + metrics', 'sync', 'emphasis'),
    ('db', 'api', 'rows', 'reply', 'default'),
    ('api', 'web', '200 JSON', 'reply', 'default'),
    ('web', 'user', 'render', 'reply', 'default'),
], columns=['source', 'target', 'label', 'style', 'variant'])
participants = {
    'user':  {'label': 'User', 'kind': 'actor'},
    'web':   {'label': 'Web app'},
    'api':   {'label': 'API'},
    'auth':  {'label': 'Auth'},
    'cache': {'label': 'Redis'},
    'db':    {'label': 'Postgres'},
}
groups = [{'label': 'Identity', 'from': 2, 'to': 3, 'color': '#e15759'},
          {'label': 'Cache fallback', 'from': 4, 'to': 7, 'color': '#4e79a7'}]
steps = [
    {'title': 'Request', 'caption': 'The user opens the dashboard; the web app calls the API.',
     'highlight': {'messages': [0, 1]}},
    {'title': 'Identity', 'caption': 'The API verifies the JWT before doing any work.',
     'highlight': {'messages': [2, 3]}},
    {'title': 'Cache miss', 'caption': 'Redis has no entry, so the API queries Postgres.',
     'highlight': {'messages': [4, 5, 6, 7]}},
    {'title': 'Response', 'caption': 'The JSON goes back and the page renders.',
     'highlight': {'messages': [8, 9], 'participants': ['user']}},
]
d3.sequence(df, participants=participants, groups=groups, steps=steps,
            notebook=True, filepath=None, figsize=[None, 650])

Participant and message properties can be edited after drawing; `d3.show()` redraws.
(The trailing `;` stops Jupyter from also printing the returned HTML.)

In [ ]:
d3.node_properties['cache']['color'] = '#e15759'
d3.edge_properties.loc[5, 'label'] = 'miss (TTL expired)'
d3.show();

## 3. Save as a standalone HTML file

Without `notebook=True` the chart is written to `filepath`; `showfig=True` opens it in the browser.
`dark_mode='auto'` follows the operating system theme until the theme button is used.

In [ ]:
d3.sequence(oauth, dark_mode='auto', filepath='sequence_oauth.html', showfig=False)
print('Saved', d3.config['filepath'])